# Residual bounds, compliant outputs and effectivity

After POD and Galerkin prediction, we ask: **how accurate is this prediction without computing the full solution again?** We first prove the bounds, then collect them in a table. The last section explains how to evaluate the certificates efficiently; it is **optional reading outside the class derivation**. Its Python API is executable here and in the notebook generated by Antora.


Prerequisites: [Galerkin reduction](https://feelpp.github.io/course-rom/rom/reduction/galerkin.html) and [dual norms and Riesz representation](https://feelpp.github.io/course-rom/rom/foundations/riesz.html). Throughout the core discussion the bilinear form is **symmetric positive definite**. See [the extension overview](https://feelpp.github.io/course-rom/rom/certification/extensions.html) for nonsymmetric and time-dependent problems.

## Two energy norms, one error

For each parameter, let $u_h\in V_h$ and $u_r\in V_r\subset V_h$ satisfy

$$
a_\mu(u_h,v)=f_\mu(v)\quad(v\in V_h),\qquad
a_\mu(u_r,v_r)=f_\mu(v_r)\quad(v_r\in V_r).
$$
Write $e=u_h-u_r$. In coordinates, $u_r=Za$: each column of $Z$ contains one reduced basis function in the full basis, and $a$ contains the reduced coefficients. Both solutions use the **same discrete model**, loads and parameter. These are reduction errors, not discretization errors relative to the continuous PDE, model discrepancy or measurement noise.
Choose a reference parameter $\bar\mu$ once. In coordinates define

| Quantity | Definition | Meaning |
| -------- | ---------- | ------- |
| Fixed reference-energy norm | $\Vert v\Vert _G^2=v^TGv$, with $G=A(\bar\mu)$ | The same measuring tool at every query; also denoted $\Vert v\Vert _V$. |
| Parameter-dependent energy norm | $\Vert v\Vert _{a_\mu}^2=a_\mu(v,v)=v^TA(\mu)v$ | The measuring tool changes with the current operator. |
| Coercivity and continuity | $\alpha=\inf_{v\ne0}a_\mu(v,v)/\Vert v\Vert _G^2$, $\gamma=\sup_{v\ne0}a_\mu(v,v)/\Vert v\Vert _G^2$ | Extremal generalized eigenvalues of $A(\mu)v=\lambda Gv$ in this symmetric case. |

**Slide notation.** The triple bars denote the current energy norm:

$$
\lvert\!\lvert\!\lvert v\rvert\!\rvert\!\rvert_\mu
:=\Vert v\Vert _{a_\mu},\qquad
\Vert v\Vert _G=\lvert\!\lvert\!\lvert v\rvert\!\rvert\!\rvert_{\bar\mu}.
$$
When the parameter is fixed locally, we also write $\lvert\!\lvert\!\lvert e\rvert\!\rvert\!\rvert$ without the subscript. Thus a bound for $\Vert e\Vert _{a_\mu}$ below is a bound for the triple-bar error; it is not a bound in the fixed reference norm. See [the notation reminder](https://feelpp.github.io/course-rom/rom/notation.html) for the correspondence with the slides.
All constants below refer to this **fixed** reference metric. A certified computation uses a verified $0<\alpha_{\mathrm{LB}}\leq\alpha$; when needed, $\gamma_{\mathrm{UB}}\geq\gamma$. The [coercivity page](https://feelpp.github.io/course-rom/rom/certification/coercivity-lower-bound.html) explains the min-Theta construction. The reference norm could also be any other fixed SPD metric, provided the constants are changed consistently.
*Proposition 1 — Equivalence of the two energy norms*\
For every $v\in V_h$,


$$
\sqrt{\alpha}\,\Vert v\Vert _G\leq\Vert v\Vert _{a_\mu}
\leq\sqrt{\gamma}\,\Vert v\Vert _G.
$$
*Proof of Proposition 1*\
The definitions of the smallest and largest Rayleigh quotients give $\alpha\Vert v\Vert _G^2\leq a_\mu(v,v)\leq\gamma\Vert v\Vert _G^2$. All quantities are nonnegative, so taking square roots proves the claim. For $G=A(\bar\mu)$ and $\mu=\bar\mu$, the two norms coincide and $\alpha=\gamma=1$.

## The residual and its two dual norms

Define the residual functional and, in coordinates, its vector:

$$
R_\mu(v)=f_\mu(v)-a_\mu(u_r,v),\qquad
\rho=f(\mu)-A(\mu)Za.
$$
*Proposition 2 — Error equation and Galerkin orthogonality*\
The error satisfies $a_\mu(e,v)=R_\mu(v)$ for all $v\in V_h$, and $a_\mu(e,v_r)=R_\mu(v_r)=0$ for every $v_r\in V_r$.
*Proof of Proposition 2*\
Subtract $a_\mu(u_r,v)$ from the full equation $a_\mu(u_h,v)=f_\mu(v)$. If $v=v_r\in V_r$, the reduced equation says the difference is zero. In coordinates this is $Z^T\rho=0$. Orthogonality only tests reduced directions: the full residual and the error can still be nonzero.
## Fixed-metric Riesz representation

Write $r_G=\Vert R_\mu\Vert _{G'}=\sup_{v\ne0}|R_\mu(v)|/\Vert v\Vert _G$. The notation $\Vert R_\mu\Vert _{V_h'}$ in other chapters means this same dual norm when $\Vert \cdot\Vert _V=\Vert \cdot\Vert _G$.
*Proposition 3 — Compute the fixed dual norm by one Riesz solve*\
Let $z_G$ solve $(z_G,v)_G=R_\mu(v)$ for every $v\in V_h$. Then


$$
Gz_G=\rho,\qquad
r_G=\Vert z_G\Vert _G=\sqrt{\rho^Tz_G}
=\sqrt{\rho^TG^{-1}\rho}.
$$
*Proof of Proposition 3*\
The metric is SPD, so the Riesz solve has a unique solution. For every $v$, Cauchy–Schwarz gives $|R_\mu(v)|=|(z_G,v)_G|\leq\Vert z_G\Vert _G\Vert v\Vert _G$. The supremum is therefore at most $\Vert z_G\Vert _G$. If $z_G\ne0$, choosing $v=z_G$ attains this value; if $z_G=0$, both norms are zero. Finally $Gz_G=\rho$ gives $\Vert z_G\Vert _G^2=z_G^TGz_G=\rho^Tz_G$.
This computes a residual norm, not the error itself: $Gz_G=\rho$ and $A(\mu)e=\rho$ are different systems except at the reference parameter. Do not replace the dual norm by $\Vert \rho\Vert _2$ unless $G=I$. Solve a linear system rather than forming an inverse.
## The energy dual norm: an exact identity, but an expensive one

Define $r_a=\Vert R_\mu\Vert _{a_\mu'}=\sup_{v\ne0}|R_\mu(v)|/\Vert v\Vert _{a_\mu}$. This uses the current energy norm, not the fixed reference norm.
*Proposition 4 — In the current energy metric, the residual norm equals the error*\
For the symmetric SPD problem,


$$
r_a=\Vert e\Vert _{a_\mu},\qquad
r_a^2=\rho^TA(\mu)^{-1}\rho.
$$
*Proof of Proposition 4*\
Proposition 2 gives $R_\mu(v)=a_\mu(e,v)$. Cauchy–Schwarz in the energy inner product bounds the quotient by $\Vert e\Vert _{a_\mu}$. Taking $v=e$ attains that value when $e\ne0$; the zero-error case is immediate. The Riesz representative in this metric is the error itself, since it solves $A(\mu)e=\rho$. Proposition 3 applied with the metric $A(\mu)$ yields the matrix identity.
This identity explains the analysis. It is **not** an efficient online estimator: applying $A(\mu)^{-1}$ generally needs a new full-dimensional solve at the query. The useful certificates below use the fixed-metric residual norm $r_G$ and inexpensive stability bounds instead.

## Certified bounds for the state error

*Proposition 5 — Two-sided bounds in the fixed reference-energy norm*\
With valid stability bounds,


$$
\frac{r_G}{\gamma_{\mathrm{UB}}}
\leq\Vert e\Vert _G
\leq\frac{r_G}{\alpha_{\mathrm{LB}}}
=: \Delta_G.
$$

The upper bound does not require a continuity upper bound; only the lower bound does.
*Proof of Proposition 5*\
For the upper bound, test the error equation with $e$:


$$
\alpha\Vert e\Vert _G^2\leq a_\mu(e,e)=R_\mu(e)
\leq r_G\Vert e\Vert _G.
$$

If $e\ne0$, divide by $\alpha\Vert e\Vert _G$ and use $\alpha_{\mathrm{LB}}\leq\alpha$. The zero-error case is immediate. For the other direction, symmetry and positivity give energy Cauchy–Schwarz; Proposition 1 then gives continuity:


$$
|R_\mu(v)|=|a_\mu(e,v)|
\leq\Vert e\Vert _{a_\mu}\Vert v\Vert _{a_\mu}
\leq\gamma\Vert e\Vert _G\Vert v\Vert _G.
$$

Take the supremum over nonzero $v$ to obtain $r_G\leq\gamma\Vert e\Vert _G\leq\gamma_{\mathrm{UB}}\Vert e\Vert _G$, and divide by $\gamma_{\mathrm{UB}}$. Together, these arguments prove $\alpha\Vert e\Vert _G\leq r_G\leq\gamma\Vert e\Vert _G$.
Other pages denote the fixed-norm certificate by $\Delta_r$; here $\Delta_r=\Delta_G$. The lower bound rules out an error smaller than its value; the upper bound certifies that the error cannot exceed its value.
*Proposition 6 — Two-sided bounds in the parameter-dependent energy norm*\
For the same fixed residual norm $r_G$, the following bounds control $\lvert\!\lvert\!\lvert e\rvert\!\rvert\!\rvert_\mu=\Vert e\Vert _{a_\mu}$:


$$
\frac{r_G}{\sqrt{\gamma_{\mathrm{UB}}}}
\leq\Vert e\Vert _{a_\mu}
\leq\frac{r_G}{\sqrt{\alpha_{\mathrm{LB}}}}
=: \Delta_a.
$$
*Proof of Proposition 6*\
The error equation and Proposition 1 give


$$
\Vert e\Vert _{a_\mu}^2=R_\mu(e)
\leq r_G\Vert e\Vert _G
\leq\frac{r_G}{\sqrt\alpha}\Vert e\Vert _{a_\mu}.
$$

Divide by the nonzero energy norm, or treat zero error separately, then replace $\alpha$ by $\alpha_{\mathrm{LB}}$. For the lower bound, for every $v$,


$$
|R_\mu(v)|\leq\Vert e\Vert _{a_\mu}\Vert v\Vert _{a_\mu}
\leq\sqrt\gamma\Vert e\Vert _{a_\mu}\Vert v\Vert _G.
$$

Taking the supremum gives $r_G\leq\sqrt\gamma\Vert e\Vert _{a_\mu}$, hence the lower bound after replacing $\gamma$ by $\gamma_{\mathrm{UB}}$. We have proved $\sqrt\alpha\Vert e\Vert _{a_\mu}\leq r_G\leq\sqrt\gamma\Vert e\Vert _{a_\mu}$.
The different denominators matter: use $\alpha_{\mathrm{LB}}$ for the fixed-norm state bound, but $\sqrt{\alpha_{\mathrm{LB}}}$ for the current-energy bound. They certify different quantities.

## Bounds for outputs: general versus compliant

*Proposition 7 — A general bounded linear output*\
Let $s_h=\ell(u_h)$ and $s_r=\ell(u_r)$, and let $\Vert \ell\Vert _{G'}$ use the fixed metric. Then


$$
|s_h-s_r|\leq\Vert \ell\Vert _{G'}\Delta_G.
$$

In coordinates, $\Vert \ell\Vert _{G'}=\sqrt{\ell^TG^{-1}\ell}$. In the current-energy metric one also has $|s_h-s_r|\leq\Vert \ell\Vert _{a_\mu'}\Delta_a$, but computing that output dual norm may itself require a parameter-dependent full solve.
*Proof of Proposition 7*\
Linearity gives $s_h-s_r=\ell(e)$. The definition of the dual norm yields $|\ell(e)|\leq\Vert \ell\Vert _{G'}\Vert e\Vert _G$. Apply Proposition 5. The energy version follows from exactly the same argument in the energy metric, using Proposition 6. The Riesz formula for the output norm is Proposition 3 with $\rho=\ell$.
*Proposition 8 — The symmetric compliant-output identity and its certificate*\
Assume the output is the **same functional as the load**: $s_h=f_\mu(u_h)$, $s_r=f_\mu(u_r)$. For conforming Galerkin reduction,


$$
s_h-s_r=\Vert e\Vert _{a_\mu}^2\geq0,
\qquad
\frac{r_G^2}{\gamma_{\mathrm{UB}}}
\leq s_h-s_r
\leq\frac{r_G^2}{\alpha_{\mathrm{LB}}}
=: \Delta_s.
$$
*Proof of Proposition 8*\
Use the full equation with $v=e$, write $u_h=u_r+e$, then use symmetry and Galerkin orthogonality:


$$
s_h-s_r=f_\mu(e)=a_\mu(u_h,e)
=a_\mu(e,e)+a_\mu(u_r,e)
=a_\mu(e,e)+a_\mu(e,u_r)
=\Vert e\Vert _{a_\mu}^2.
$$

The last term vanishes since $u_r\in V_r$. This proves both the identity and nonnegativity. Square the two inequalities of Proposition 6; all terms are nonnegative. In particular $\Delta_s=\Delta_a^2$, not $\Delta_G^2$ in general.
**Check compliance before using the quadratic result.** In our two-parameter bar, $f_{\mu,g}=b+g e_n$. Its compliant output is $(b+g e_n)^Tu$. The mean temperature $b^Tu$ is compliant only when $g=0$; otherwise it is the general output of Proposition 7. Its error need not equal an energy-error square.

## Effectivity: how pessimistic is the certificate?

For nonzero errors define three different ratios:

$$
\eta_G=\frac{\Delta_G}{\Vert e\Vert _G},\qquad
\eta_a=\frac{\Delta_a}{\Vert e\Vert _{a_\mu}},\qquad
\eta_s=\frac{\Delta_s}{s_h-s_r}\quad\text{(compliant output)}.
$$
A value of 1 means an exact bound; a value of 10 means a tenfold overestimate of the corresponding error. Comparing a bound in one norm with an error in another does not define the intended effectivity.
*Proposition 9 — Effectivity bounds in both norms and for the compliant output*\
For $e\ne0$ and a valid positive coercivity lower bound,


$$
1\leq\frac{\alpha}{\alpha_{\mathrm{LB}}}
\leq\eta_G\leq\frac{\gamma}{\alpha_{\mathrm{LB}}},
\qquad
1\leq\sqrt{\frac{\alpha}{\alpha_{\mathrm{LB}}}}
\leq\eta_a\leq\sqrt{\frac{\gamma}{\alpha_{\mathrm{LB}}}}.
$$

For the compliant output,


$$
\eta_s=\eta_a^2,\qquad
1\leq\frac{\alpha}{\alpha_{\mathrm{LB}}}
\leq\eta_s\leq\frac{\gamma}{\alpha_{\mathrm{LB}}}.
$$

A known $\gamma_{\mathrm{UB}}$ can replace $\gamma$ in all upper bounds.
*Proof of Proposition 9*\
Proposition 5 proved $\alpha\Vert e\Vert _G\leq r_G\leq\gamma\Vert e\Vert _G$. Divide by $\alpha_{\mathrm{LB}}\Vert e\Vert _G$ to obtain the first chain. Proposition 6 proved $\sqrt\alpha\Vert e\Vert _{a_\mu}\leq r_G\leq\sqrt\gamma\Vert e\Vert _{a_\mu}$. Divide by $\sqrt{\alpha_{\mathrm{LB}}}\Vert e\Vert _{a_\mu}$ for the second. Proposition 8 gives $s_h-s_r=\Vert e\Vert _{a_\mu}^2$ and $\Delta_s=\Delta_a^2$, so the compliant-output effectivity is exactly $\eta_a^2$. Squaring the second chain proves its bounds.
Effectivity requires the full-order reference error and is therefore a **validation diagnostic**, not an online quantity. The certificate itself does not need that reference. At zero error the residual and bounds are zero in exact arithmetic, but the ratios are undefined. Near roundoff, report absolute errors and bounds instead of unstable ratios. A ratio below 1 signals a violated assumption or implementation/numerical error, apart from a declared tolerance.
For a general linear output, the ratio of Proposition 7&#8217;s bound to $|\ell(e)|$ has no analogous finite upper bound from coercivity alone: the output error can vanish while the state error does not.
## A small example: the error direction matters

Take $A=\operatorname{diag}(1,4)$ and $G=I$, with $\alpha_{\mathrm{LB}}=\alpha=1$ and $\gamma=4$. Both coordinate errors below can arise from Galerkin using the other coordinate as its reduced space.

| Error | $r_G$ | $\Vert e\Vert _G$ | $\Vert e\Vert _a$ | $\eta_G$ | $\eta_a$ | $\eta_s$ |
| ----- | ----- | ----------------- | ----------------- | -------- | -------- | -------- |
| $(1,0)^T$ | 1 | 1 | 1 | 1 | 1 | 1 |
| $(0,1)^T$ | 4 | 1 | 2 | 4 | 2 | 4 |

Even the exact coercivity constant does not make every certificate exact: it describes the least stable direction. Replacing $\alpha_{\mathrm{LB}}$ by $\alpha_{\mathrm{LB}}/10$ multiplies $\Delta_G,\Delta_s,\eta_G,\eta_s$ by 10, and $\Delta_a,\eta_a$ by $\sqrt{10}$. It changes the bounds, not the predictions.

## Related approximation results: what the residual does not replace

These results explain approximation quality but do not by themselves give a new-parameter online certificate. They complete the summary of the Galerkin/POD results already studied.
*Proposition 10 — Galerkin best approximation and the fixed-norm estimate*\
Let $d_G=\inf_{v_r\in V_r}\Vert u_h-v_r\Vert _G$. For symmetric SPD Galerkin,


$$
\Vert e\Vert _{a_\mu}=\min_{v_r\in V_r}\Vert u_h-v_r\Vert _{a_\mu},
\qquad \Vert e\Vert _G\leq\sqrt{\gamma/\alpha}\,d_G.
$$

The general coercive Céa estimate is $\Vert e\Vert _G\leq(\gamma/\alpha)d_G$; it does not require symmetry. In that nonsymmetric version, $\gamma$ denotes the continuity constant in $|a_\mu(v,w)|\leq\gamma\Vert v\Vert _G\Vert w\Vert _G$, not just a maximum diagonal Rayleigh quotient.
*Proof of Proposition 10*\
For any $v_r$, decompose $u_h-v_r=e+(u_r-v_r)$. The two terms are energy-orthogonal by Proposition 2 and symmetry. Thus


$$
\Vert u_h-v_r\Vert _{a_\mu}^2=\Vert e\Vert _{a_\mu}^2+\Vert u_r-v_r\Vert _{a_\mu}^2.
$$

This proves optimality, with equality at $v_r=u_r$. For every $v_r$, Proposition 1 gives $\Vert e\Vert _G\leq\Vert e\Vert _{a_\mu}/\sqrt\alpha\leq\Vert u_h-v_r\Vert _{a_\mu}/\sqrt\alpha\leq\sqrt{\gamma/\alpha}\Vert u_h-v_r\Vert _G$. Take the infimum. For the general Céa estimate, orthogonality gives $\alpha\Vert e\Vert _G^2\leq a_\mu(e,e)=a_\mu(e,u_h-v_r)\leq\gamma\Vert e\Vert _G\Vert u_h-v_r\Vert _G$. Divide for nonzero error and take the infimum. This last argument only needs continuity and coercivity.
*Proposition 11 — The POD identity concerns training projection error*\
For fixed metric $G=LL^T$, positive snapshot weights $W=\operatorname{diag}(w_j)$, and singular values of $Y=L^TSW^{1/2}$, the POD basis minimizes


$$
E_{\mathrm{POD}}^2=\sum_jw_j\Vert u_j-ZZ^TGu_j\Vert _G^2
=\sum_{i>r}\sigma_i^2,\qquad Z^TGZ=I_r.
$$
*Proof of Proposition 11*\
Set $B=L^TZ$, so $B^TB=I_r$. The objective becomes $\Vert Y-BB^TY\Vert _F^2=\Vert Y\Vert _F^2-\Vert B^TY\Vert _F^2$. Expand in a full orthonormal left singular basis $U_i$, including zero singular values. The captured term is $\sum_i\sigma_i^2\Vert B^TU_i\Vert _2^2$. These weights lie between 0 and 1 and sum to $r$, so this sum is at most $\sum_{i=1}^r\sigma_i^2$. Taking $B=U_{:,1:r}$ attains the maximum; subtract it from total energy. The [POD chapter](https://feelpp.github.io/course-rom/rom/reduction/pod.html) illustrates the construction on the parametrized PDE.
This is a weighted average over the training snapshots. It is not a maximum over parameter space and not the error of a Galerkin prediction at an unseen parameter.

## Summary — Which quantity does each result control?

All rows use the same fixed metric, residual $r_G$ and valid stability bounds. Every identity/bound below was proved above; an exact expression may be mathematically useful without being cheap to evaluate.

| Quantity | Identity or bound | Result and interpretation |
| -------- | ----------------- | ------------------------- |
| Fixed-norm state error | $r_G/\gamma_{\mathrm{UB}}\leq\Vert e\Vert _G\leq\Delta_G=r_G/\alpha_{\mathrm{LB}}$ | Proposition 5. Computable certificate once the fixed dual norm and stability bounds are available. |
| Current-energy state error | $r_G/\sqrt{\gamma_{\mathrm{UB}}}\leq\Vert e\Vert _{a_\mu}\leq\Delta_a=r_G/\sqrt{\alpha_{\mathrm{LB}}}$ | Proposition 6. Different norm, different denominator. |
| Energy-dual identity | $\Vert e\Vert _{a_\mu}=r_a=\sqrt{\rho^TA(\mu)^{-1}\rho}$ | Proposition 4. Exact, but a direct evaluation needs a full solve with the current operator. |
| General linear output | $\lvert s_h-s_r\rvert\leq\Vert \ell\Vert _{G'}\Delta_G$ | Proposition 7. No compliance assumption; the output error need not be nonnegative. |
| Compliant output | $s_h-s_r=\Vert e\Vert _{a_\mu}^2$, $r_G^2/\gamma_{\mathrm{UB}}\leq s_h-s_r\leq\Delta_s=r_G^2/\alpha_{\mathrm{LB}}$ | Proposition 8. Requires symmetry, compliance and conforming Galerkin. |
| Fixed-norm effectivity | $1\leq\eta_G\leq\gamma/\alpha_{\mathrm{LB}}$ | Proposition 9. A validation diagnostic requiring the actual error. |
| Energy effectivity | $1\leq\eta_a\leq\sqrt{\gamma/\alpha_{\mathrm{LB}}}$ | Proposition 9. Uses the energy error in its denominator. |
| Compliant-output effectivity | $\eta_s=\eta_a^2$, $1\leq\eta_s\leq\gamma/\alpha_{\mathrm{LB}}$ | Proposition 9. Quadratic error also squares the energy effectivity. |
| Galerkin best approximation | $\Vert e\Vert _{a_\mu}=\min_{v_r}\Vert u_h-v_r\Vert _{a_\mu}$, $\Vert e\Vert _G\leq\sqrt{\gamma/\alpha}\,d_G$ | Proposition 10. A priori comparison; the best-approximation error is not known online. |
| General coercive Céa estimate | $\Vert e\Vert _G\leq(\gamma/\alpha)d_G$ | Proposition 10. Does not need symmetry. |
| POD training projection | $E_{\mathrm{POD}}^2=\sum_{i>r}\sigma_i^2$ | Proposition 11. Training-data identity, not an unseen-parameter certificate. |


## Check your understanding


1. Why do the denominators differ between the two state certificates?
1. For a symmetric compliant output, why is the output error nonnegative? Which orthogonality is used?
1. For the two-parameter bar, when is mean temperature compliant?
1. Which quantities require a full reference solve to evaluate: a certificate, its effectivity, or both?

Continue with the [certification practical](https://feelpp.github.io/course-rom/course-rom/labs/session03-error-estimation.html) or the [sampling/Galerkin practical](https://feelpp.github.io/course-rom/course-rom/labs/galerkin-sampling.html). The following implementation is supplementary reading, not an additional required derivation in class.
## Optional reading — How do we make prediction and certification efficient?

A direct residual calculation forms $\rho=f-AZa$ and solves $Gz_G=\rho$. That is a valid implementation of the preceding proofs, but still depends on the full dimension $n$. Efficient prediction alone does not automatically make certification efficient.
## Affine separation: parameter dependence in scalar coefficients

Assume fixed matrices and vectors with known scalar functions:

$$
A(\mu)=\sum_{q=1}^{Q_A}\theta_q^A(\mu)A_q,\qquad
f(\mu)=\sum_{q=1}^{Q_f}\theta_q^f(\mu)f_q.
$$
For the bar, $A=\mu K+M$ and $f=b+g e_n$: the operator weights are $(\mu,1)$ and the load weights are $(1,g)$. The metric remains $G=K+M$ at every query. Affine components need not individually be SPD; the assembled operator must satisfy our SPD assumptions.
Offline, project once $A_{r,q}=Z^TA_qZ$ and $f_{r,q}=Z^Tf_q$. Online, assemble only small arrays and solve

$$
\left(\sum_q\theta_q^A A_{r,q}\right)a
=\sum_q\theta_q^f f_{r,q}.
$$
This is a superposition of fixed **operators and loads**. The full solution is generally **not** an affine superposition of solutions for each operator: the inverse of a sum is not the sum of the inverses.
## Superposition of fixed Riesz solutions

For each fixed load and each fixed operator action on a basis vector, solve once

$$
G\zeta_q^f=f_q,\qquad G\zeta_{q,i}^A=A_q z_i.
$$
Since the **fixed** Riesz map is linear, the representative of the residual is exactly

$$
z_G(\mu)=\sum_q\theta_q^f(\mu)\zeta_q^f
-\sum_{q=1}^{Q_A}\sum_{i=1}^r\theta_q^A(\mu)a_i(\mu)\zeta_{q,i}^A.
$$
This is the useful superposition of precomputed solutions: it concerns Riesz representatives. Reconstructing this full vector online would still cost work proportional to $n$. Instead, precompute enough inner products to evaluate its norm from small coefficients alone.
Collect the fixed columns and online coefficients in the **same order**:

$$
B=[f_1\ \cdots\ f_{Q_f}\ A_1Z\ \cdots\ A_{Q_A}Z],\qquad
c=\begin{pmatrix}\theta^f\\-\theta_1^A a\\ \vdots\\-\theta_{Q_A}^A a\end{pmatrix},
\qquad\rho=Bc.
$$
There are $p=Q_f+Q_Ar$ columns. With $D=G^{-1}B$, their Gram matrix is $H=B^TD$, giving $r_G^2=c^THc$. Compute the solves offline; no inverse should be formed explicitly. Near tiny residuals this quadratic expression can lose accuracy through cancellation.
## A QR factor avoids the final Gram quadratic form

Use the Cholesky convention $G=LL^T$. Solve $LY=B$ and compute a reduced QR factorization $Y=QT$. Then

$$
r_G=\Vert L^{-1}\rho\Vert _2=\Vert Yc\Vert _2
=\Vert QTc\Vert _2=\Vert Tc\Vert _2.
$$
The identity uses $Q^TQ=I$. The fixed matrix $T$ has $\min(n,p)$ rows and $p$ columns. In the intended regime $p\ll n$, it is small. Dependent residual columns are allowed; avoid dividing by small QR diagonal entries. QR reduces the cancellation risk of the squared Gram expression, but does not eliminate floating-point limits when the true residual is nearly zero.

| Stage | Work | Dependence on full dimension |
| ----- | ---- | ---------------------------- |
| Offline | Project affine operators/loads, factor $G$, transform residual columns and form $T$. Precompute a fixed output vector and its dual norm if desired. | Allowed: snapshots and factorizations are expensive, but reused. |
| Online prediction | Evaluate scalar affine coefficients, assemble and solve the $r\times r$ reduced system. | No $n$-dimensional arrays. |
| Online certification | Build $c$, compute $\Vert Tc\Vert _2$, evaluate a verified $\alpha_{\mathrm{LB}}$, then return $\Delta_G,\Delta_a,\Delta_s$. | No full residual and no Riesz solve at the query. |
| Validation only | Solve the full model, measure true errors and calculate effectivities. | Full-dimensional work; never part of the proposed online certificate. |

For dense reduced algebra the query costs approximately $O(Q_Ar^2+Q_fr+r^3+p^2)$. This avoids $n$ only when the affine counts and reduced/residual dimensions stay small. Evaluating a full reconstructed field still costs $O(nr)$. A cheap verified coercivity lower bound remains essential; affine decomposition alone does not provide it. Rebuild the offline data when the reduced basis, affine components or fixed metric change.
## A small Python API: prepare once, query many times

The following NumPy-only API separates offline preparation from online prediction/certification. It returns bounds, not effectivities: those require reference errors. It handles several affine load terms, including the flux. An optional fixed linear output is treated separately from the compliant output.


In [ ]:
import numpy as np


class AffineCertificate:
    """Offline preparation for a fixed SPD metric and a fixed Galerkin basis."""

    def __init__(self, operators, loads, basis, metric, output_vector=None):
        Z = np.asarray(basis, dtype=float)
        G = np.asarray(metric, dtype=float)
        As = tuple(np.asarray(A, dtype=float) for A in operators)
        fs = tuple(np.asarray(f, dtype=float) for f in loads)
        if Z.ndim != 2 or not 0 < Z.shape[1] <= Z.shape[0]:
            raise ValueError('basis must have shape (n, r), with 0 < r <= n')
        n, r = Z.shape
        if G.shape != (n, n) or not np.allclose(G, G.T):
            raise ValueError('metric must be a symmetric n-by-n matrix')
        if not As or any(A.shape != (n, n) for A in As):
            raise ValueError('supply at least one n-by-n affine operator')
        if not fs or any(f.shape != (n,) for f in fs):
            raise ValueError('supply at least one length-n affine load')
        if not all(np.all(np.isfinite(v)) for v in (Z, G, *As, *fs)):
            raise ValueError('offline arrays must be finite')
        if any(not np.allclose(A, A.T) for A in As):
            raise ValueError('this API covers symmetric affine operators')
        L = np.linalg.cholesky(G)
        np.linalg.cholesky(Z.T @ G @ Z)  # Require independent basis columns.
        self._Ar = np.stack([Z.T @ A @ Z for A in As])
        self._Fr = np.column_stack([Z.T @ f for f in fs])
        B = np.column_stack([*fs, *[A @ Z for A in As]])
        weighted_columns = np.linalg.solve(L, B)
        _, self._T = np.linalg.qr(weighted_columns, mode='reduced')
        self._ellr = None
        if output_vector is not None:
            ell = np.asarray(output_vector, dtype=float)
            if ell.shape != (n,) or not np.all(np.isfinite(ell)):
                raise ValueError('output_vector must be a finite length-n vector')
            self._ellr = Z.T @ ell
            self._ell_norm = np.linalg.norm(np.linalg.solve(L, ell))
        # Keep only reduced operators, loads, output and residual factor.
        self.offline_sizes = {'n': n, 'r': r, 'residual_columns': B.shape[1],
                              'factor_shape': self._T.shape}

    def query(self, theta_a, theta_f, alpha_lb, gamma_ub=None):
        """Online only. Caller supplies valid SPD/stability assumptions."""
        ta, tf = np.asarray(theta_a, dtype=float), np.asarray(theta_f, dtype=float)
        if ta.shape != (len(self._Ar),) or tf.shape != (self._Fr.shape[1],):
            raise ValueError('one scalar coefficient is needed per affine term')
        if not np.all(np.isfinite(ta)) or not np.all(np.isfinite(tf)):
            raise ValueError('affine coefficients must be finite')
        if not np.isfinite(alpha_lb) or alpha_lb <= 0:
            raise ValueError('alpha_lb must be a verified positive lower bound')
        if gamma_ub is not None and (not np.isfinite(gamma_ub) or gamma_ub < alpha_lb):
            raise ValueError('gamma_ub must be positive and at least alpha_lb')
        Ar = np.einsum('q,qij->ij', ta, self._Ar)
        fr = self._Fr @ tf
        # A small Cholesky solve checks reduced SPD; full-model SPD is an assumption.
        C = np.linalg.cholesky(Ar)
        a = np.linalg.solve(C.T, np.linalg.solve(C, fr))
        c = np.concatenate([tf, *[-coefficient*a for coefficient in ta]])
        residual = np.linalg.norm(self._T @ c)
        answer = {'a': a, 'residual_norm': residual,
                  'delta_G': residual / alpha_lb,
                  'delta_a': residual / np.sqrt(alpha_lb),
                  'delta_s': residual**2 / alpha_lb,
                  'compliant_output': float(fr @ a)}
        if self._ellr is not None:
            answer['linear_output'] = float(self._ellr @ a)
            answer['delta_output'] = self._ell_norm * answer['delta_G']
        if gamma_ub is not None:
            answer.update(lower_G=residual / gamma_ub,
                          lower_a=residual / np.sqrt(gamma_ub),
                          lower_s=residual**2 / gamma_ub)
        return answer


The API checks sizes, finiteness and positivity, but cannot prove online that a supplied coercivity lower bound is valid for the full model. Nor does reduced-matrix SPD establish full-model SPD. Those are mathematical input assumptions. The min-Theta formula in the following example is justified by the positive thermal affine decomposition.
## Example — the same two-parameter bar and a POD basis

Use a small mesh and a rank-three energy POD basis for the demonstration. All full solves in this cell are **offline training**. The API will return both the compliant output and the fixed mean-temperature output; they are different when the imposed flux is nonzero.


In [ ]:
# Offline thermal model and training snapshots.
n = 40
h = 1.0 / n
K = (2*np.eye(n) - np.eye(n, k=1) - np.eye(n, k=-1)) / h
K[-1, -1] = 1.0 / h
mass = np.full(n, h)
mass[-1] = h / 2
M = np.diag(mass)
b = mass.copy()
e_right = np.eye(n)[:, -1]
G = K + M  # Reference diffusivity mu_bar = 1.
training = [(mu, g) for mu in np.geomspace(0.1, 10, 6)
            for g in (-0.25, 0.0, 0.25)]
S = np.column_stack([np.linalg.solve(mu*K + M, b + g*e_right)
                     for mu, g in training])
L = np.linalg.cholesky(G)
U, sigma, _ = np.linalg.svd(L.T @ S / np.sqrt(len(training)), full_matrices=False)
Z = np.linalg.solve(L.T, U[:, :3])
certificate = AffineCertificate((K, M), (b, e_right), Z, G, output_vector=b)
print('Offline sizes:', certificate.offline_sizes)

# Online query: no full matrix, full residual, full solution or Riesz solve.
mu, g = 0.35, 0.12
result = certificate.query(theta_a=(mu, 1.0), theta_f=(1.0, g),
                           alpha_lb=min(mu, 1.0), gamma_ub=max(mu, 1.0))
print(f"Online compliant output: {result['compliant_output']:.6e}, "
      f"bound: {result['delta_s']:.6e}")
print(f"Online mean temperature: {result['linear_output']:.6e}, "
      f"general-output bound: {result['delta_output']:.6e}")


For $G=K+M$, both $K$ and $M$ are positive semidefinite and

$$
\min(\mu,1)G\preceq\mu K+M\preceq\max(\mu,1)G.
$$
This proves the lower/upper stability choices used by the API. Their evaluation does not require an eigensolve. The flux changes the load, not the operator stability constants.
## Validation — compare the API to full solves and inspect effectivity

Only this final validation cell computes full reference solutions and direct residual norms. These operations are not part of `query`. It checks both norms, both outputs, the exact energy-dual identity, the compliant-output identity, the lower/upper bounds and the effectivity relation. The two norm-dependent scalings should not be conflated.


In [ ]:
import matplotlib.pyplot as plt

test_mu = np.unique(np.r_[np.geomspace(0.12, 8.0, 15), 1.0])
errors_G, errors_a, errors_s = [], [], []
bounds_G, bounds_a, bounds_s = [], [], []
print('mu | error G / certificate | error energy / certificate | eta G / eta a / eta s')
for mu in test_mu:
    g = 0.12
    alpha_lb, gamma_ub = min(mu, 1.0), max(mu, 1.0)
    ans = certificate.query((mu, 1.0), (1.0, g), alpha_lb, gamma_ub)
    A, f = mu*K + M, b + g*e_right
    uh, ur = np.linalg.solve(A, f), Z @ ans['a']
    error, rho = uh - ur, f - A @ ur
    err_G = np.sqrt(error @ G @ error)
    err_a = np.sqrt(error @ A @ error)
    err_s = float(f @ uh - ans['compliant_output'])
    direct_residual = np.sqrt(rho @ np.linalg.solve(G, rho))
    exact_energy_dual = np.sqrt(rho @ np.linalg.solve(A, rho))
    assert np.isclose(ans['residual_norm'], direct_residual, rtol=1e-7, atol=1e-12)
    assert np.isclose(exact_energy_dual, err_a, rtol=1e-7, atol=1e-12)
    assert np.isclose(err_s, err_a**2, rtol=1e-6, atol=1e-12)
    for actual, low, high in ((err_G, ans['lower_G'], ans['delta_G']),
                              (err_a, ans['lower_a'], ans['delta_a']),
                              (err_s, ans['lower_s'], ans['delta_s'])):
        assert low <= actual + 1e-11 and actual <= high + 1e-11
    assert abs(b @ uh - ans['linear_output']) <= ans['delta_output'] + 1e-11
    eta_G, eta_a = ans['delta_G'] / err_G, ans['delta_a'] / err_a
    # Use the squared energy error for the ratio near cancellation in output differences.
    eta_s = ans['delta_s'] / err_a**2
    assert np.isclose(eta_s, eta_a**2)
    assert 1 - 1e-7 <= eta_G <= gamma_ub / alpha_lb + 1e-7
    assert 1 - 1e-7 <= eta_a <= np.sqrt(gamma_ub / alpha_lb) + 1e-7
    errors_G.append(err_G); errors_a.append(err_a); errors_s.append(err_a**2)
    bounds_G.append(ans['delta_G']); bounds_a.append(ans['delta_a']); bounds_s.append(ans['delta_s'])
    print(f'{mu:.3f} | {err_G:.3e} / {ans["delta_G"]:.3e} | '
          f'{err_a:.3e} / {ans["delta_a"]:.3e} | {eta_G:.2f} / {eta_a:.2f} / {eta_s:.2f}')
print('Offline/online certification: verified norms, outputs, bounds and effectivities.')

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, errors, bounds, label in zip(
        axes, (errors_G, errors_a, errors_s), (bounds_G, bounds_a, bounds_s),
        ('Fixed reference-energy error', 'Current-energy error', 'Compliant-output error')):
    ax.loglog(test_mu, errors, 'o-', label='Reference error')
    ax.loglog(test_mu, bounds, 's--', label='Online certificate')
    ax.set(xlabel='Diffusivity mu', ylabel=label)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


Change the query diffusivity and flux, or rebuild with a different POD rank. At $\mu=1$, $A(\mu)=G$ and the stability bounds equal 1: all three state/compliant certificates are exact up to roundoff. For exact zero errors, do not calculate effectivity ratios. General-output bounds can remain pessimistic even at that reference parameter.
The next methodological steps are [greedy basis selection](https://feelpp.github.io/course-rom/rom/reduction/greedy.html) and [time-dependent prediction](https://feelpp.github.io/course-rom/rom/certification/extensions.html). A greedy algorithm can call the same online certificate API; it need not implement the residual factorization itself.
